Importamos librerias

In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, inspect

Instalamos la libreria si falta ( opcional)

In [2]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 47.4 MB/s eta 0:00:00


Añadidos la info que nos requiere el código

In [3]:
usuario = "root"
password = "Sg!1992!sg"
host = "127.0.0.1"
puerto = "3306"
bbdd = "proyecto_jupiter_alchemy"

engine = create_engine(
    f"mysql+mysqlconnector://{usuario}:{password}@{host}:{puerto}/{bbdd}"
)

Visualizamos las tablas que tenemos en el SCHEMA en el  MySQL Workbench

In [4]:

pd.read_sql("SHOW TABLES;", con=engine)

InterfaceError: (mysql.connector.errors.InterfaceError) 2003: Can't connect to MySQL server on '127.0.0.1:3306' (Errno 111: Connection refused)
(Background on this error at: https://sqlalche.me/e/20/rvf5)

Añadimos la ruta donde tenemos los CSVs ( nota guardar el notebook en la misma carpeta)

In [ ]:
RUTA_CSV = Path("..") / "CSVs" / "CSVs principales"
RUTA_CSV

WindowsPath('../CSVs/CSVs principales')

Comprabamos que existe la ruta

In [ ]:
RUTA_CSV.exists()

True

Haremos una prueba primero con un solo CSV. En este caso el ETIQUETAS
Haremos posteriormente los mismos dos pasos anteriores para comprobarla ruta de etiquetas.csv

In [ ]:
ruta_etiquetas = RUTA_CSV / "etiquetas.csv"
ruta_etiquetas

WindowsPath('../CSVs/CSVs principales/etiquetas.csv')

In [ ]:
ruta_etiquetas.exists()

True

Creamos una dataframe con el contenido del csv

In [ ]:
df_etiquetas = pd.read_csv(ruta_etiquetas, encoding="utf-8")
df_etiquetas.head()

,id_t_id,t_id
0,1,Apple 1.png
1,2,Apple 10.png
2,3,Apple 100.png
3,4,Apple 101.png
4,5,Apple 102.png


Observamos la forma

In [ ]:
df_etiquetas.shape

(69687, 2)

Observamos el nombre de las columnas

In [ ]:
df_etiquetas.columns.tolist()

['id_t_id', 't_id']

Lo mismo que pero miramos el nombre que tiene en MySQL

In [ ]:

inspector = inspect(engine)
columnas_mysql_etiquetas = [col["name"] for col in inspector.get_columns("etiquetas")]
columnas_mysql_etiquetas

['id_t_id', 't_id']

Comprobamos que no tenemos nada cargado en la tabla en MySQL

In [ ]:
filas_antes = pd.read_sql("SELECT COUNT(*) AS total FROM etiquetas", con=engine)
filas_antes

,total
0,0


Realizamos la carga

In [ ]:
df_etiquetas.to_sql(
    name="etiquetas",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

69687

Comprobamos si han cargados las filas en MySQL

In [ ]:
filas_despues = pd.read_sql("SELECT COUNT(*) AS total FROM etiquetas", con=engine)
filas_despues

,total
0,69687


Creamos una función para la carga masiva de los datos

In [ ]:

def cargar_tabla(tabla, nombre_csv, engine, ruta_base_csv, chunksize=5000):
    print(f"\n--- Cargando tabla: {tabla} ---")

    ruta_csv = ruta_base_csv / nombre_csv
    print(f"Ruta CSV: {ruta_csv}")

    if not ruta_csv.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta_csv}")

    df = pd.read_csv(ruta_csv, encoding="utf-8")
    print(f"Shape del CSV: {df.shape}")

    columnas_csv = df.columns.tolist()
    columnas_mysql = [col["name"] for col in inspector.get_columns(tabla)]

    print("Columnas CSV:  ", columnas_csv)
    print("Columnas MySQL:", columnas_mysql)

    if columnas_csv != columnas_mysql:
        raise ValueError(f"Las columnas no coinciden en la tabla '{tabla}'")

    filas_antes = pd.read_sql(f"SELECT COUNT(*) AS total FROM {tabla}", con=engine)
    print("Filas antes:")
    print(filas_antes)

    df.to_sql(
        name=tabla,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=chunksize,
        method="multi"
    )

    filas_despues = pd.read_sql(f"SELECT COUNT(*) AS total FROM {tabla}", con=engine)
    print("Filas después:")
    print(filas_despues)

    print(f"Tabla '{tabla}' cargada correctamente.")

Creamos una primera lista con las tablas dimensiones

In [ ]:
tablas_dimensiones = [
    ("proveedores", "proveedores.csv"),
    ("clientes", "clientes.csv"),
    ("frutas", "frutas.csv"),
    ("marca", "marca.csv"),
]

Con un bucle recorremos la lista de dimensiones y ejecutamos la función

In [ ]:
for tabla, archivo in tablas_dimensiones:
    cargar_tabla(tabla, archivo, engine, RUTA_CSV)

Realizamos el mismo proceso con la tablas de hechos

In [ ]:
tablas_hechos = [
    ("lotes", "lotes.csv"),
    ("compras", "compras.csv"),
    ("ventas", "ventas.csv"),
]

In [ ]:
for tabla, archivo in tablas_hechos:
    cargar_tabla(tabla, archivo, engine, RUTA_CSV)

Con un bucle hacemos una consulta a la cantidad de filas en cada tabla

In [ ]:
for tabla in ["proveedores", "clientes", "etiquetas", "frutas", "marca", "lotes", "compras", "ventas"]:
    total = pd.read_sql(f"SELECT COUNT(*) AS total FROM {tabla}", con=engine)
    print(f"{tabla}:")
    print(total)
    print("-" * 40)

proveedores:
   total
0     35
----------------------------------------
clientes:
   total
0     34
----------------------------------------
etiquetas:
   total
0  69687
----------------------------------------
frutas:
   total
0     15
----------------------------------------
marca:
   total
0     35
----------------------------------------
lotes:
   total
0  69687
----------------------------------------
compras:
   total
0  70549
----------------------------------------
ventas:
   total
0  70549
----------------------------------------
